# Krish Naik Agentic AI 3.0 Course Langchain-Runtime

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage,HumanMessage,ToolMessage
from langchain.agents.middleware import before_model,before_agent,after_model, after_agent,wrap_model_call,wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import AgentState

from langgraph.runtime import Runtime
from langchain.tools import ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from dataclasses import dataclass
from typing import Any,Callable

from langchain.agents.middleware import dynamic_prompt
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

## Agent State

In [19]:
class MyAgentState(AgentState):
    user_name: str

@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    print(f"[TOOL] get_weather({location})")
    return f"The weather in {location} is sunny."

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    state_schema=MyAgentState,
)
result = agent.invoke({"messages": [{ "role": "user", "content": "What is the weather in Tokyo?"} ],
    "user_name": "John"
})
from rich import print
print(result)

[TOOL] get_weather(Tokyo)

{
    'messages': [
        HumanMessage(
            content='What is the weather in Tokyo?',
            additional_kwargs={},
            response_metadata={},
            id='4cd52b39-d105-48aa-b09e-132a3e1ccf9a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 132,
                    'total_tokens': 155,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wGAGNKtkphO8FNQBASFUNV1gCC',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02f96-7445-75e1-9d44-82d6ca703401-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_vtrFizK1IJEu75FoylgDng4s',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 132,
                'output_tokens': 23,
                'total_tokens': 155,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='The weather in Tokyo is sunny.',
            name='get_weather',
            id='fd1a53aa-dc96-4937-8f5b-9396444dc34e',
            tool_call_id='call_vtrFizK1IJEu75FoylgDng4s'
        ),
        AIMessage(
            content="It's sunny in Tokyo right now. Would you like more details (temperature, humidity, hourly 
forecast, or a multi-day outlook)?",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 99,
                    'prompt_tokens': 167,
                    'total_tokens': 266,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wIZPmc9UgDaW8InDgBySDpHQ7L',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02f96-7951-7771-bdd8-927f055ab9c7-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 167,
                'output_tokens': 99,
                'total_tokens': 266,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        )
    ],
    'user_name': 'John'
}

## Agent context and Tool runtime

In [20]:
class MyAgentState(AgentState):
    user_name: str


@dataclass
class MyCustomerContext:
    user_id: str

loyalty_store=InMemoryStore()
checkpointer = InMemorySaver()

@tool
def save_customer_preferences(preference: str, runtime: ToolRuntime[MyCustomerContext]) -> str:
    """Save a customer's preference to long-term memory."""
    print(f' save_customer_preferences runtime.state={runtime.state}')
    user_id = runtime.context.user_id
    print(f'save_customer_preferences called for {user_id} with incoming {preference}')
    # Look for existing customer record
    existing = runtime.store.get( ("users"),user_id)

    if existing:
       
        data = existing.value
    else:
        data = {}

    print(f'saving preference {preference} for {user_id}')
    # Add/update preference
    data["preferences"] = preference

    # Save it
    runtime.store.put( ("users"), user_id,data)
    return f"Saved preference: {preference}"

    
@tool
def fetch_customer_preferences(runtime: ToolRuntime[MyCustomerContext])-> str:
  "Fetch the customer's Saved Preferences from a long term Memory"
  print(f' fetch_customer_preferences runtime.state={runtime.state}')
  user_id=runtime.context.user_id
  print(f'fetch_customer_preferences called for {user_id}')
  preferences = "No Preferences"

  if runtime.store:
    if memory := runtime.store.get(('users'),user_id):
      preferences = memory.value['preferences']

  return preferences

pref_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[save_customer_preferences,fetch_customer_preferences],
    context_schema=MyCustomerContext,
    store=loyalty_store,
    checkpointer = checkpointer,
    state_schema=MyAgentState
)

config = {"configurable": {"thread_id": "pref-agent-demo"}}

user_id = "aj123"
dialogue = [
   "What's my fav movie?",
    "I am AJ, I like movie interstellar",
   "What's my fav movie?",
]
state_value = {'user_name': 'Ajuser'}
for d in dialogue: 
    result = pref_agent.invoke(
        {"messages": [{"role": "user", "content": d}] ,
        "user_name" :  'Ajuser'
        },
        context=MyCustomerContext(user_id=user_id),   # injected at invocation time
        config = config
    )

from rich import print
print(result)

## Another user
user_id = "priya123"
result = pref_agent.invoke(
        {"messages": [{"role": "user", "content": "What's my fav movie?"}],
        "user_name" :  'priyauser'
        },
        context=MyCustomerContext(user_id=user_id),   # injected at invocation time
        config = config,
)
print(result)

fetch_customer_preferences runtime.state={'messages': [HumanMessage(content="What's my fav movie?", 
additional_kwargs={}, response_metadata={}, id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'), AIMessage(content='', 
additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 
153, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 
'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0', tool_calls=[{'name': 'fetch_customer_preferences', 'args': {},
'id': 'call_klf0D6PklcTsYCRHrT2F3eFX', 'type': 'tool_call'}], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 153, 'output_tokens': 20, 'total_tokens': 173, 'input_token_details': {'audio': 0, 
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})], 'user_name': 'Ajuser'}

fetch_customer_preferences called for aj123

save_customer_preferences runtime.state={'messages': [HumanMessage(content="What's my fav movie?", 
additional_kwargs={}, response_metadata={}, id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'), AIMessage(content='', 
additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 
153, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 
'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0', tool_calls=[{'name': 'fetch_customer_preferences', 'args': {},
'id': 'call_klf0D6PklcTsYCRHrT2F3eFX', 'type': 'tool_call'}], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 153, 'output_tokens': 20, 'total_tokens': 173, 'input_token_details': {'audio': 0, 
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), ToolMessage(content='No Preferences', 
name='fetch_customer_preferences', id='2d983da9-656b-4401-bc56-501668e15bff', 
tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'), AIMessage(content="I don't know yet — I couldn't find a saved 
favorite movie for you. What is your favorite movie? If you tell me, I can remember it for next time. If you're not
sure, tell me a genre or a few movies you like and I can suggest likely favorites.", additional_kwargs={'refusal': 
None}, response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 181, 'total_tokens': 502, 
'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 
'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 
'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': 
None, 'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh', 'service_tier': 'default', 'finish_reason': 'stop', 
'logprobs': None}, id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0', tool_calls=[], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 181, 'output_tokens': 321, 'total_tokens': 502, 'input_token_details': {'audio': 0,
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}}), HumanMessage(content='I am AJ, I like 
movie interstellar', additional_kwargs={}, response_metadata={}, id='500d1b38-5e4a-44fd-9208-6754a2fb691e'), 
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 
159, 'prompt_tokens': 256, 'total_tokens': 415, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 
'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': 
{'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wsctjdbL30P84pOYZjKiPrR65J', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f97-0820-7e41-af28-5bf906cd6e69-0', tool_calls=[{'name': 'save_customer_preferences', 'args': 
{'preference': 'AJ: favorite movie: Interstellar'}, 'id': 'call_HJEL49lJmg4231OtkMoXLBv7', 'type': 'tool_call'}], 
invalid_tool_calls=[], usage_metadata={'input_tokens': 256, 'output_tokens': 159, 'total_tokens': 415, 
'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})], 
'user_name': 'Ajuser'}

save_customer_preferences called for aj123 with incoming AJ: favorite movie: Interstellar

saving preference AJ: favorite movie: Interstellar for aj123

fetch_customer_preferences runtime.state={'messages': [HumanMessage(content="What's my fav movie?", 
additional_kwargs={}, response_metadata={}, id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'), AIMessage(content='', 
additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 
153, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 
'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0', tool_calls=[{'name': 'fetch_customer_preferences', 'args': {},
'id': 'call_klf0D6PklcTsYCRHrT2F3eFX', 'type': 'tool_call'}], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 153, 'output_tokens': 20, 'total_tokens': 173, 'input_token_details': {'audio': 0, 
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), ToolMessage(content='No Preferences', 
name='fetch_customer_preferences', id='2d983da9-656b-4401-bc56-501668e15bff', 
tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'), AIMessage(content="I don't know yet — I couldn't find a saved 
favorite movie for you. What is your favorite movie? If you tell me, I can remember it for next time. If you're not
sure, tell me a genre or a few movies you like and I can suggest likely favorites.", additional_kwargs={'refusal': 
None}, response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 181, 'total_tokens': 502, 
'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 
'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 
'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': 
None, 'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh', 'service_tier': 'default', 'finish_reason': 'stop', 
'logprobs': None}, id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0', tool_calls=[], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 181, 'output_tokens': 321, 'total_tokens': 502, 'input_token_details': {'audio': 0,
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}}), HumanMessage(content='I am AJ, I like 
movie interstellar', additional_kwargs={}, response_metadata={}, id='500d1b38-5e4a-44fd-9208-6754a2fb691e'), 
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 
159, 'prompt_tokens': 256, 'total_tokens': 415, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 
'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': 
{'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wsctjdbL30P84pOYZjKiPrR65J', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f97-0820-7e41-af28-5bf906cd6e69-0', tool_calls=[{'name': 'save_customer_preferences', 'args': 
{'preference': 'AJ: favorite movie: Interstellar'}, 'id': 'call_HJEL49lJmg4231OtkMoXLBv7', 'type': 'tool_call'}], 
invalid_tool_calls=[], usage_metadata={'input_tokens': 256, 'output_tokens': 159, 'total_tokens': 415, 
'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}), 
ToolMessage(content='Saved preference: AJ: favorite movie: Interstellar', name='save_customer_preferences', 
id='ae2174da-351b-4e28-b982-59f3bfa52d7c', tool_call_id='call_HJEL49lJmg4231OtkMoXLBv7'), AIMessage(content='Got 
it, AJ — I saved Interstellar as your favorit

fetch_customer_preferences called for aj123

{
    'messages': [
        HumanMessage(
            content="What's my fav movie?",
            additional_kwargs={},
            response_metadata={},
            id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 153,
                    'total_tokens': 173,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0',
            tool_calls=[
                {
                    'name': 'fetch_customer_preferences',
                    'args': {},
                    'id': 'call_klf0D6PklcTsYCRHrT2F3eFX',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 153,
                'output_tokens': 20,
                'total_tokens': 173,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='No Preferences',
            name='fetch_customer_preferences',
            id='2d983da9-656b-4401-bc56-501668e15bff',
            tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'
        ),
        AIMessage(
            content="I don't know yet — I couldn't find a saved favorite movie for you. What is your favorite 
movie? If you tell me, I can remember it for next time. If you're not sure, tell me a genre or a few movies you 
like and I can suggest likely favorites.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 321,
                    'prompt_tokens': 181,
                    'total_tokens': 502,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 181,
                'output_tokens': 321,
                'total_tokens': 502,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        HumanMessage(
            content='I am AJ, I like movie interstellar',
            additional_kwargs={},
            respo

{
    'messages': [
        HumanMessage(
            content="What's my fav movie?",
            additional_kwargs={},
            response_metadata={},
            id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 153,
                    'total_tokens': 173,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0',
            tool_calls=[
                {
                    'name': 'fetch_customer_preferences',
                    'args': {},
                    'id': 'call_klf0D6PklcTsYCRHrT2F3eFX',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 153,
                'output_tokens': 20,
                'total_tokens': 173,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='No Preferences',
            name='fetch_customer_preferences',
            id='2d983da9-656b-4401-bc56-501668e15bff',
            tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'
        ),
        AIMessage(
            content="I don't know yet — I couldn't find a saved favorite movie for you. What is your favorite 
movie? If you tell me, I can remember it for next time. If you're not sure, tell me a genre or a few movies you 
like and I can suggest likely favorites.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 321,
                    'prompt_tokens': 181,
                    'total_tokens': 502,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 181,
                'output_tokens': 321,
                'total_tokens': 502,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        HumanMessage(
            content='I am AJ, I like movie interstellar',
            additional_kwargs={},
            respo

In [33]:
class MyAgentState(AgentState):
    logged_in_user: str
    
@dataclass
class MyContext:
    user_id: str

@tool
def get_weather(location: str, runtime: ToolRuntime) -> str:
    """get weather"""
    print(f"\nget_weather {location}")
    return f"Weather is sunny 22F for {location}"

@before_agent
def my_before_agent(state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
    print("[ my_before_agent ]",state, runtime)
    return None

@wrap_model_call
def my_wrap_model_call(request, handler):
    print("[ my_wrap_model_call ] ",request.state, request.runtime)
    return handler(request)

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    middleware = [ my_before_agent, my_wrap_model_call],
    context_schema=MyContext,
    store=InMemoryStore(),
    state_schema=MyAgentState
)

result = agent.invoke({"messages": [("user", "what is the weather in tokyo ? ")], "logged_in_user": "aj123"})
from rich import print
print(result)

[ my_before_agent ]
{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        )
    ],
    'logged_in_user': 'aj123'
}
Runtime(
    context=None,
    store=<langgraph.store.memory.InMemoryStore object at 0x1192c9f40>,
    stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x1192a2020>,
    heartbeat=<function _no_op_heartbeat at 0x1166574c0>,
    previous=None,
    execution_info=ExecutionInfo(
        checkpoint_id='1f19f1ad-67f0-6f02-8000-3775da90c4b5',
        checkpoint_ns='my_before_agent.before_agent:9b8162a5-67fe-575d-725b-762450485ccc',
        task_id='9b8162a5-67fe-575d-725b-762450485ccc',
        thread_id=None,
        run_id=None,
        node_attempt=1,
        node_first_attempt_time=1787507304.1817842
    ),
    server_info=None,
    control=<langgraph.runtime.RunControl object at 0x118c6ef20>
)

[ my_wrap_model_call ] 
{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        )
    ],
    'logged_in_user': 'aj123'
}
Runtime(
    context=None,
    store=<langgraph.store.memory.InMemoryStore object at 0x1192c9f40>,
    stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x1192a2020>,
    heartbeat=<function _no_op_heartbeat at 0x1166574c0>,
    previous=None,
    execution_info=ExecutionInfo(
        checkpoint_id='1f19f1ad-6810-6780-8001-64acaa6c56b2',
        checkpoint_ns='model:c2c26a83-c52c-e573-49c2-cccac2d2a2f9',
        task_id='c2c26a83-c52c-e573-49c2-cccac2d2a2f9',
        thread_id=None,
        run_id=None,
        node_attempt=1,
        node_first_attempt_time=1787507304.194535
    ),
    server_info=None,
    control=<langgraph.runtime.RunControl object at 0x118c6ef20>
)

get_weather Tokyo

[ my_wrap_model_call ] 
{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 87,
                    'prompt_tokens': 130,
                    'total_tokens': 217,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG6auohR7XjC1fvARtEZA9527Lpdb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02fbc-e713-7372-863d-3bf1fd0fdc0f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_Q2cGfMiSszz7vqVZGPJkXNHp',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 130,
                'output_tokens': 87,
                'total_tokens': 217,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='Weather is sunny 22F for Tokyo',
            name='get_weather',
            id='875cceee-b6cb-48d2-81ef-ecb6cee09609',
            tool_call_id='call_Q2cGfMiSszz7vqVZGPJkXNHp'
        )
    ],
    'logged_in_user': 'aj123'
}
Runtime(
    context=None,
    store=<langgraph.store.memory.InMemoryStore object at 0x1192c9f40>,
    stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x1192a2020>,
    heartbeat=<function _no_op_heartbeat at 0x1166574c0>,
    previous=None,
    execution_info=ExecutionInfo(
        checkpoint_id='1f19f1ad-7bcf-6f28-8003-8f727ca90e31',
        checkpoint_ns='model:a170bfd7-0f5c-bb43-4cd1-c7042d6f5b49',
        task_id='a170bfd7-0f5c-bb43-4cd1-c7042d6f5b49',
        thread_id=None,
        run_id=None,
        node_attempt=1,
        node_first_attempt_time=1787507306.265217
    ),
    server_info=None,
    control=<langgraph.runtime.RunControl object at 0x118c6ef20>
)

{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 87,
                    'prompt_tokens': 130,
                    'total_tokens': 217,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG6auohR7XjC1fvARtEZA9527Lpdb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02fbc-e713-7372-863d-3bf1fd0fdc0f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_Q2cGfMiSszz7vqVZGPJkXNHp',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 130,
                'output_tokens': 87,
                'total_tokens': 217,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='Weather is sunny 22F for Tokyo',
            name='get_weather',
            id='875cceee-b6cb-48d2-81ef-ecb6cee09609',
            tool_call_id='call_Q2cGfMiSszz7vqVZGPJkXNHp'
        ),
        AIMessage(
            content='Right now in Tokyo it’s sunny with a temperature of 22°F (about −5.6°C). \n\nIf you want, I 
can:\n- give a multi-day forecast\n- show hourly temperatures or precipitation chances\n- convert to a different 
unit or compare to your local weather\n\nWhich would you like?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 264,
                    'prompt_tokens': 166,
                    'total_tokens': 430,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG6awIfat5QGv1bE9CWwGhMgqiBfK',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02fbc-ef2a-7190-bcdc-e2b7792a55b7-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 166,
                'output_tokens': 264,
                'total_tokens': 430,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        )
    ],
    'logged_in_user': 'aj123'
}

## Dynamic prompting

In [39]:
@dataclass
class WeatherContext:
    country: str


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"The weather in {location} is 75 degrees."


@dynamic_prompt
def weather_prompt(request):
    country = request.runtime.context.country

    print(" DYNAMIC PROMPT EXECUTED")
    print("country =", country)

    if country == "US":
        prompt = (
            "You are a weather assistant. "
            "Report temperatures in Fahrenheit."
        )
    else:
        prompt = (
            "You are a weather assistant. "
            "Report temperatures in Celsius."
        )

    print("prompt =", prompt)

    return prompt


agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    context_schema=WeatherContext,
    middleware=[weather_prompt],
)

result = agent.invoke({"messages": [("user", "what is the weather in tokyo ? ")]},context = WeatherContext(country= "US"))
from rich import print
print(result["messages"][-1])

result = agent.invoke({"messages": [("user", "what is the weather in tokyo ? ")]},context = WeatherContext(country= "IND"))
from rich import print
print(result["messages"][-1])

DYNAMIC PROMPT EXECUTED

country = US

prompt = You are a weather assistant. Report temperatures in Fahrenheit.

DYNAMIC PROMPT EXECUTED

country = US

...

prompt = You are a weather assistant. Report temperatures in Celsius.

DYNAMIC PROMPT EXECUTED

country = IND

prompt = You are a weather assistant. Report temperatures in Celsius.

AIMessage(
    content='The current temperature in Tokyo is about 24°C (the source reported 75°F, which converts to ≈24°C). 
Would you like more details (conditions, humidity, wind, or a forecast)?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 435,
            'prompt_tokens': 190,
            'total_tokens': 625,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 384,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-EG6oWR1LlFg4yRT1gORu1P9kznyBM',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a02fc9-c6d7-7871-ad64-a4a395b43b9f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 190,
        'output_tokens': 435,
        'total_tokens': 625,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 384}
    }
)

# human in the loop middleware with interrupt and command

In [19]:
@tool
def cancel_booking(booking_id:str) -> str:
    """cancel the booking for a booking id"""
    return f"Cancelled the booking for {booking_id}"

@tool
def send_booking_confirmation(booking_id:str) -> str:
    """send the booking confirmation for a booking id"""
    return f"booking confirmed for for {booking_id}"
    
agent = create_agent(
    model = "openai:gpt-5-mini", 
    tools=[cancel_booking, send_booking_confirmation],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "cancel_booking": True,               # all four decisions allowed (default config)
                "send_booking_confirmation": False,    # safe operation, auto-approved, never pauses
            },
            description_prefix="CineBot action pending your approval",
        ),
    ],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "hitl-demo-approve"}}
## a modified version of invoke for HITL
result = agent.invoke(
    {"messages": [("user", "Cancel booking BK1042")]},
    config=config,
    version="v2",   # the current, recommended invoke pattern for reading interrupts
)
from rich import print
print(type(result))
print(result["messages"][-1].content)
print(result.interrupts)

print('approval send with invoke')
# Resume with approval decision
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config, # Same thread ID to resume the paused conversation
    version="v2",
)
print(type(result))
print(result["messages"][-1].content)



print('-----reject hitl example------')
config = {"configurable": {"thread_id": "hitl-demo-reject"}}
## a modified version of invoke for HITL
result = agent.invoke(
    {"messages": [("user", "Cancel booking BK1042")]},
    config=config,
    version="v2",   # the current, recommended invoke pattern for reading interrupts
)
from rich import print
print(result["messages"][-1].content)
print(type(result))
print(result.interrupts)

print('reject send with invoke')
# Resume with reject decision
result = agent.invoke(
    Command(resume={"decisions": [{"type": "reject"}]}),
    config=config, # Same thread ID to resume the paused conversation
    version="v2",
)
print(result["messages"][-1].content)
print(type(result))

<class 'langgraph.types.GraphOutput'>

/var/folders/w7/bjnp92015253vpp0xtyzp16c0000gn/T/ipykernel_35444/1266041888.py:35: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


(
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'description': "CineBot action pending your approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                }
            ],
            'review_configs': [
                {'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}
            ]
        },
        id='1ada0c5a0ccbde62e99fdf0bba0b8f95'
    ),
)

approval send with invoke

<class 'langgraph.types.GraphOutput'>

...

(
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'description': "CineBot action pending your approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                }
            ],
            'review_configs': [
                {'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}
            ]
        },
        id='47770e4417a2da2007c6453d5ecdac0c'
    ),
)

reject send with invoke

/var/folders/w7/bjnp92015253vpp0xtyzp16c0000gn/T/ipykernel_35444/1266041888.py:68: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


I can do that — I haven’t cancelled it yet because I didn’t get your confirmation. Cancelling BK1042 now is 
irreversible and may affect refunds or incur fees depending on the provider’s policy.

Please confirm you want me to cancel this booking. Reply with one of:
- “Yes, cancel BK1042” — proceed with cancellation and send confirmation
- “Don’t cancel” — keep the booking

If you want, tell me also whether you’d like me to check refund/fee details or help rebook instead.

<class 'langgraph.types.GraphOutput'>

### respond example

In [29]:
@tool
def ask_customer(question: str) -> str:
    """Ask the customer a clarifying question and wait for their reply."""
    return "placeholder -- never actually reached, respond intercepts this"


ask_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[ask_customer],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"ask_customer": {"allowed_decisions": ["respond"]}})],
    checkpointer=InMemorySaver(),
)

config_ask = {"configurable": {"thread_id": "hitl-demo-3"}}
result = ask_agent.invoke({"messages": [("user", "Cancel booking BK1042")]}, config=config_ask)

from rich import print
print(result)

print('invoke with respond action again')
result = ask_agent.invoke(
    Command(resume={"decisions": [{
        "type": "respond",
        "message": "Its Booking BK1042, for the movie interstellar",
    }]}),
    config=config_ask
)
print(result)

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='3ba941fd-a82b-45e0-afc5-1fb18a9155e3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 258,
                    'prompt_tokens': 136,
                    'total_tokens': 394,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGV9sGPARsPG3Zto91Yen2jVaisQm',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0355d-ad3a-7331-9228-dddec74b02e2-0',
            tool_calls=[
                {
                    'name': 'ask_customer',
                    'args': {
                        'question': 'Do you want me to cancel booking BK1042 now? Please confirm and provide the 
name on the booking or the email used for it, and tell me if you want information about refunds/penalties before I 
proceed.'
                    },
                    'id': 'call_76zyGFsekeMjm3zqtxXoeM2B',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 136,
                'output_tokens': 258,
                'total_tokens': 394,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'ask_customer',
                        'args': {
                            'question': 'Do you want me to cancel booking BK1042 now? Please confirm and provide 
the name on the booking or the email used for it, and tell me if you want information about refunds/penalties 
before I proceed.'
                        },
                        'description': "Tool execution requires approval\n\nTool: ask_customer\nArgs: {'question': 
'Do you want me to cancel booking BK1042 now? Please confirm and provide the name on the booking or the email used 
for it, and tell me if you want information about refunds/penalties before I proceed.'}"
                    }
                ],
                'review_configs': [{'action_name': 'ask_customer', 'allowed_decisions': ['respond']}]
            },
            id='da2d76650a1da64ab3789db12251c944'
        )
    ]
}

invoke with respond action again

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='3ba941fd-a82b-45e0-afc5-1fb18a9155e3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 258,
                    'prompt_tokens': 136,
                    'total_tokens': 394,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGV9sGPARsPG3Zto91Yen2jVaisQm',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0355d-ad3a-7331-9228-dddec74b02e2-0',
            tool_calls=[
                {
                    'name': 'ask_customer',
                    'args': {
                        'question': 'Do you want me to cancel booking BK1042 now? Please confirm and provide the 
name on the booking or the email used for it, and tell me if you want information about refunds/penalties before I 
proceed.'
                    },
                    'id': 'call_76zyGFsekeMjm3zqtxXoeM2B',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 136,
                'output_tokens': 258,
                'total_tokens': 394,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        ),
        ToolMessage(
            content='Its Booking BK1042, for the movie interstellar',
            name='ask_customer',
            id='8ad039d7-b3c2-42e4-b9be-d2c435210b14',
            tool_call_id='call_76zyGFsekeMjm3zqtxXoeM2B'
        ),
        AIMessage(
            content='Got it — booking BK1042 for the movie Interstellar.\n\nI can cancel this for you. Quick 
confirmation/details I need before I proceed:\n1. Do you want me to cancel it now? (yes/no)  \n2. The name on the 
booking or the email used for the booking.  \n3. Which cinema or ticketing platform was used (e.g., AMC, Cineworld,
Fandango, Ticketmaster, etc.) — if you know.\n\nAlso tell me if you want me to first check the cancellation/refund 
policy (possible fees, refunds as credit vs. money back) before cancelling.\n\nIf you prefer to cancel it yourself,
say so and I’ll give step-by-step instructions for the common platforms.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 668,
                    'prompt_tokens': 218,
                    'total_tokens': 886,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 512,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGV9yrcukibWrWJYs3ApxEcdrqxrd',
                'service_tier': 'default',
 

## edit action

In [30]:
@tool
def ask_customer(question: str) -> str:
    """Ask the customer a clarifying question and wait for their reply."""
    return "placeholder -- never actually reached, respond intercepts this"

    
ask_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[ask_customer],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"ask_customer": {"allowed_decisions": ["respond","edit"]}})],
    checkpointer=InMemorySaver(),
)

config_ask = {"configurable": {"thread_id": "hitl-demo-4"}}
result = ask_agent.invoke({"messages": [("user", "Cancel booking BK1042")]}, config=config_ask)

from rich import print
print(result)

print('invoke with edit action again')
result = ask_agent.invoke(
    Command(resume={"decisions": [{
                    "type": "edit",
                    "edited_action": {
                        "name": "ask_customer",
                        "args": {
                            "question": "Can you confirm that you want to cancel booking BK1042 for Interstellar?"
                        }
                    }
                }]}),
    config=config_ask
)
print(result)

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='898053c7-3b89-407f-b7e1-d0665d81efe3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 383,
                    'prompt_tokens': 136,
                    'total_tokens': 519,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 320,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGVB6bO4KgFmBmr04qnL9hPcCXlHq',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0355e-d9bc-7681-a888-128cd163ba29-0',
            tool_calls=[
                {
                    'name': 'ask_customer',
                    'args': {
                        'question': 'Please confirm: do you want me to cancel booking BK1042? Reply "Confirm" to 
proceed now, or reply "Check fees" if you want me to check cancellation fees and refund amount first.'
                    },
                    'id': 'call_iKE0PToLzqGqdW4HRBQ87tCc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 136,
                'output_tokens': 383,
                'total_tokens': 519,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 320}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'ask_customer',
                        'args': {
                            'question': 'Please confirm: do you want me to cancel booking BK1042? Reply "Confirm" 
to proceed now, or reply "Check fees" if you want me to check cancellation fees and refund amount first.'
                        },
                        'description': 'Tool execution requires approval\n\nTool: ask_customer\nArgs: 
{\'question\': \'Please confirm: do you want me to cancel booking BK1042? Reply "Confirm" to proceed now, or reply 
"Check fees" if you want me to check cancellation fees and refund amount first.\'}'
                    }
                ],
                'review_configs': [{'action_name': 'ask_customer', 'allowed_decisions': ['respond', 'edit']}]
            },
            id='b60426a28be663d4161608501f531787'
        )
    ]
}

invoke with edit action again

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='898053c7-3b89-407f-b7e1-d0665d81efe3'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 383,
                    'prompt_tokens': 136,
                    'total_tokens': 519,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 320,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGVB6bO4KgFmBmr04qnL9hPcCXlHq',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0355e-d9bc-7681-a888-128cd163ba29-0',
            tool_calls=[
                {
                    'type': 'tool_call',
                    'name': 'ask_customer',
                    'args': {
                        'question': 'Can you confirm that you want to cancel booking BK1042 for Interstellar?'
                    },
                    'id': 'call_iKE0PToLzqGqdW4HRBQ87tCc'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 136,
                'output_tokens': 383,
                'total_tokens': 519,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 320}
            }
        ),
        ToolMessage(
            content='placeholder -- never actually reached, respond intercepts this',
            name='ask_customer',
            id='0f0aa603-c574-4f70-94f1-8a6eb14e7f4c',
            tool_call_id='call_iKE0PToLzqGqdW4HRBQ87tCc'
        ),
        AIMessage(
            content='I can help with that. Do you want me to proceed and cancel booking BK1042 now?\n\nIf yes, I 
need one of these for verification before I cancel:\n- the email used for the booking, or\n- the last 4 digits of 
the payment card on the booking, or\n- the phone number associated with the booking.\n\nIf you’d rather cancel it 
yourself, tell me which platform you booked on (website/app/third party) and I’ll give step‑by‑step 
instructions.\n\nAlso tell me if you need any special handling (refund requested, exchange, or if you booked a 
non‑refundable fare).',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 585,
                    'prompt_tokens': 189,
                    'total_tokens': 774,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 448,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGVBBbuBaJ3LwmMIx8zRIztVfGSUT',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a0355e-ef29-72b1-bf72-e0dbec902c5e-0',
            tool_calls=[],


## conditional interrupt

In [35]:
from langchain.agents.middleware import ToolCallRequest

@tool
def cancel_booking(bookingid: str, amount: int) -> str:
    """cancel booking"""
    return f"booking for {amount} cancelled {bookingid}" 

    
def too_large_amount(request: ToolCallRequest) -> bool:
    """Pause writes to paths outside the workspace directory."""
    amount = request.tool_call["args"].get("amount", "")
    return amount > 50 #interrupted only for $50 or move request



agent = create_agent(
    model="gpt-5.5",
    tools=[cancel_booking],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={ "cancel_booking": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": too_large_amount,
                }
            },
        ),
    ]
)

config = {"configurable": {"thread_id": "hitl-demo-conditional-interrupt"}}
result = agent.invoke({"messages": [("user", "Cancel booking BK1042 for $20")]})

from rich import print
print(result)

result = agent.invoke({"messages": [("user", "Cancel booking BK1043 for $70")]})

from rich import print
print(result)

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1042 for $20',
            additional_kwargs={},
            response_metadata={},
            id='c188cabb-3d60-487c-a710-3b9dbb1d5c3f'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 134,
                    'total_tokens': 158,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGVsjvzbn2iUBiSK1PVLBSmYK8IKg',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a03588-231c-7171-b3ea-4852241e3fde-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'bookingid': 'BK1042', 'amount': 20},
                    'id': 'call_C363iR95cek3RHFiV9AOeR8A',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 134,
                'output_tokens': 24,
                'total_tokens': 158,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='booking for 20 cancelled BK1042',
            name='cancel_booking',
            id='3a01b88b-1d30-4855-8390-d1d4bdf405e8',
            tool_call_id='call_C363iR95cek3RHFiV9AOeR8A'
        ),
        AIMessage(
            content='Booking BK1042 has been cancelled for $20.',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 14,
                    'prompt_tokens': 177,
                    'total_tokens': 191,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGVslvugVnA9EGOgPppejgsaU14p0',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a03588-2b21-7ca0-99bb-dcd0b56abf8e-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 177,
                'output_tokens': 14,
                'total_tokens': 191,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ]
}

{
    'messages': [
        HumanMessage(
            content='Cancel booking BK1043 for $70',
            additional_kwargs={},
            response_metadata={},
            id='aca0d506-3b60-4971-8c35-326f2226ff91'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 134,
                    'total_tokens': 158,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.5-2026-04-23',
                'system_fingerprint': None,
                'id': 'chatcmpl-EGVso7WSAnkkFzUQn1jQnwkvK82ts',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a03588-33a3-7cc3-b1c2-c897f8991e1c-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'bookingid': 'BK1043', 'amount': 70},
                    'id': 'call_whyuvTa74pwPhGmOl8ST9bIs',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 134,
                'output_tokens': 24,
                'total_tokens': 158,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'bookingid': 'BK1043', 'amount': 70},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'bookingid': 'BK1043', 'amount': 70}"
                    }
                ],
                'review_configs': [
                    {'action_name': 'cancel_booking', 'allowed_decisions': ['approve', 'edit', 'reject']}
                ]
            },
            id='503eb7df6a8dff5140598211c95c1f2c'
        )
    ]
}